# 🎯 VieNeu-TTS LoRA Fine-tuning v2 (Optimized for Kaggle T4)

**Cải tiến so với v1:**
- ✅ Dynamic Padding (không padding cố định 2048 → tiết kiệm 60-70% VRAM)
- ✅ Cosine Learning Rate Scheduler (thay vì linear)
- ✅ Early Stopping (patience=5 để tự dừng khi overfit)
- ✅ LoRA mở rộng tới MLP layers (gate_proj, up_proj, down_proj)
- ✅ Learning rate giảm 50% (5e-5 thay vì 1e-4)
- ✅ Dropout tăng gấp đôi (0.1 thay vì 0.05)
- ✅ Val split 15% thay vì 5%
- ✅ Cell test inference cuối notebook

**Thời gian dự kiến:** ~2.5 giờ trên Kaggle T4 GPU

In [ ]:
# 1. Cài đặt thư viện huấn luyện
!pip install -q transformers peft torch datasets librosa soundfile tqdm phonemizer sea-g2p
!pip install -q git+https://github.com/Neuphonic/NeuCodec.git
!pip install -U bitsandbytes

In [ ]:
from sea_g2p import Normalizer, G2P

print("⚙️ Đang khởi tạo bộ xử lý Tiếng Việt (sea-g2p)...")
norm = Normalizer()
g2p_engine = G2P()

def normalize_text(text):
    return norm.normalize(text)

def phonemize_with_dict(text):
    return g2p_engine.convert(text)

print("Text Engine đã sẵn sàng!")

In [ ]:
import os
import torch
import librosa
import json
import re
import pandas as pd
from neucodec import NeuCodec
from tqdm import tqdm
from huggingface_hub import login

# ⚠️ THAY BẰNG TOKEN CỦA BẠN
login("hf_YOUR_HUGGINGFACE_TOKEN_HERE")

# ==========================================
# 🎯 1. CẤU HÌNH ĐẦU VÀO & NGƯỜI NÓI
# ==========================================
INPUT_DATA_DIR = "/kaggle/input/datasets/phmngcduy/data-utterances" 
TARGET_SPEAKER = "SPEAKER_00"

OUTPUT_DIR = "/kaggle/working/dataset_encoded"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METADATA_PATH = os.path.join(INPUT_DATA_DIR, "metadata.csv")
ENCODED_PATH = os.path.join(OUTPUT_DIR, "metadata_encoded.csv")

# ==========================================
# 🛠️ 2. MÃ HÓA VÀ CHẨN ĐOÁN LỖI TẬN GỐC
# ==========================================
print(f"🔍 Bắt đầu KIỂM TRA LỖI từ: {INPUT_DATA_DIR}")

df_full = pd.read_csv(METADATA_PATH)
df_target = df_full[df_full['speaker_id'] == TARGET_SPEAKER]

print(f"✅ Đã tìm thấy {len(df_target)} câu nói của {TARGET_SPEAKER} trong file CSV. Bắt đầu mã hóa...")

device = "cuda" if torch.cuda.is_available() else "cpu"
codec = NeuCodec.from_pretrained("neuphonic/neucodec").to(device)
codec.eval()

valid_samples = []

# --- BỘ ĐẾM LỖI ---
err_path = 0
err_dur = 0
err_text = 0
err_librosa = 0
err_encode = 0

for index, row in tqdm(df_target.iterrows(), total=len(df_target), desc="Đang mã hóa"):
    filename = str(row['audio_file'])
    duration_sec = float(row['duration_sec'])
    text = str(row['text']).strip()
    
    audio_path = os.path.join(INPUT_DATA_DIR, filename)
    
    # 1. Kiểm tra file có tồn tại thật trong ổ cứng không?
    if not os.path.exists(audio_path):
        err_path += 1
        continue
        
    # 2. [CẢI TIẾN v2] Thu hẹp bộ lọc: chỉ nhận file 3-10 giây (sweet spot)
    if not (3.0 <= duration_sec <= 10.0):
        err_dur += 1
        continue
        
    # 3. Kiểm tra văn bản rỗng
    if not text:
        err_text += 1
        continue
        
    # Tự động thêm dấu chấm để AI dễ đọc
    if text[-1] not in ".,?!":
        text += "."
        
    # 4. Thử đọc file âm thanh
    try:
        wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    except Exception as e:
        if err_librosa == 0: print(f"\n❌ Phát hiện file audio hỏng ({filename}): {e}")
        err_librosa += 1
        continue

    # 5. Thử mã hóa bằng NeuCodec
    try:
        wav_tensor = torch.from_numpy(wav).float().unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            codes = codec.encode_code(wav_tensor)
            codes = codes.squeeze(0).squeeze(0).cpu().numpy().flatten().tolist()
            codes = [int(x) for x in codes]
        
        if codes and all(0 <= c < 65536 for c in codes):
            valid_samples.append(f"{filename}|{text}|{json.dumps(codes)}\n")
        else:
            err_encode += 1
            
    except Exception as e:
        if err_encode == 0: print(f"\n❌ Lỗi quá tải/Tensor ({filename}): {e}")
        err_encode += 1
        continue

with open(ENCODED_PATH, 'w', encoding='utf-8') as f:
    f.writelines(valid_samples)

print(f"\n🎉 HOÀN TẤT! Đã mã hóa thành công: {len(valid_samples)} file.")
print(f"📉 BẢNG BÁO CÁO LỖI (Số file bị vứt bỏ):")
print(f"  - Mất do KHÔNG TÌM THẤY file .wav: {err_path} file")
print(f"  - Mất do thời lượng ngoài 3-10s: {err_dur} file")
print(f"  - Mất do không có văn bản: {err_text} file")
print(f"  - Mất do file .wav bị lỗi/hỏng: {err_librosa} file")
print(f"  - Mất do lỗi mô hình NeuCodec: {err_encode} file")

In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, 
    default_data_collator, EarlyStoppingCallback
)
from transformers.trainer_utils import get_last_checkpoint

# ==========================================
# 1. TIỀN XỬ LÝ — DYNAMIC PADDING (Cải tiến v2)
# ==========================================
def preprocess_sample(sample, tokenizer):
    """Tiền xử lý KHÔNG padding — để DataCollator tự pad theo batch."""
    speech_gen_start = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_START|>')
    speech_gen_end = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_END|>')
    ignore_index = -100
    
    # Thêm 5 token im lặng vào đuôi
    silence_tokens = [0] * 5  
    extended_codes = sample["codes"] + silence_tokens

    chat = (
        f"<|TEXT_PROMPT_START|>{sample['phones']}<|TEXT_PROMPT_END|>"
        f"<|SPEECH_GENERATION_START|>"
        + "".join([f"<|speech_{i}|>" for i in extended_codes])
        + "<|SPEECH_GENERATION_END|>"
    )
    
    raw_ids = tokenizer.encode(chat)
    
    # [v2] Giới hạn max 2048 nhưng KHÔNG PAD
    raw_ids = raw_ids[:2048]
    
    input_ids = torch.tensor(raw_ids, dtype=torch.long)
    labels = torch.full_like(input_ids, ignore_index)
    
    speech_gen_start_idx = (input_ids == speech_gen_start).nonzero(as_tuple=True)[0]
    speech_gen_end_idx = (input_ids == speech_gen_end).nonzero(as_tuple=True)[0]
    
    if len(speech_gen_start_idx) > 0 and len(speech_gen_end_idx) > 0:
        start_pos = speech_gen_start_idx[0]
        end_pos = speech_gen_end_idx[0] + 1 
        labels[start_pos:end_pos] = input_ids[start_pos:end_pos]
        
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


class DynamicPaddingCollator:
    """Pad các sample trong batch tới chiều dài dài nhất trong batch đó.
    Tiết kiệm rất nhiều VRAM so với padding cố định 2048."""
    
    def __init__(self, pad_token_id, ignore_index=-100):
        self.pad_token_id = pad_token_id
        self.ignore_index = ignore_index
    
    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        
        batch_input_ids = []
        batch_labels = []
        batch_attention_mask = []
        
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            
            input_ids = torch.cat([f["input_ids"], torch.full((pad_len,), self.pad_token_id, dtype=torch.long)])
            labels = torch.cat([f["labels"], torch.full((pad_len,), self.ignore_index, dtype=torch.long)])
            attention_mask = torch.cat([f["attention_mask"], torch.zeros(pad_len, dtype=torch.long)])
            
            batch_input_ids.append(input_ids)
            batch_labels.append(labels)
            batch_attention_mask.append(attention_mask)
        
        return {
            "input_ids": torch.stack(batch_input_ids),
            "labels": torch.stack(batch_labels),
            "attention_mask": torch.stack(batch_attention_mask)
        }


class VieNeuDataset(Dataset):
    def __init__(self, metadata_path, tokenizer):
        self.samples = []
        self.tokenizer = tokenizer
        with open(metadata_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('|')
                if len(parts) >= 3:
                    self.samples.append({"text": parts[1], "codes": json.loads(parts[2])})
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        return preprocess_sample({"phones": phonemize_with_dict(sample["text"]), "codes": sample["codes"]}, self.tokenizer)

# ==========================================
# 2. KHỞI TẠO MODEL — LoRA mở rộng MLP (v2)
# ==========================================
MODEL_ID = "pnnbao-ump/VieNeu-TTS-v2"
print(f"Đang tải mô hình gốc: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")

# [v2] LoRA config cải tiến: thêm MLP, tăng dropout
lora_config = LoraConfig(
    r=16,                    # Giữ r=16 (đủ cho voice style transfer)
    lora_alpha=32,           # alpha = 2×r (chuẩn)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"     # [v2] Thêm MLP layers
    ],
    lora_dropout=0.1,        # [v2] Tăng dropout: 0.05 → 0.1
    bias="none", 
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==========================================
# 3. DATASET — Val split 15% (v2)
# ==========================================
full_dataset = VieNeuDataset(ENCODED_PATH, tokenizer)
print(f"📊 Tổng số mẫu hợp lệ: {len(full_dataset)}")

# [v2] 15% val thay vì 5%
val_size = max(10, int(0.15 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_data, eval_data = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # [v2] Seed cố định cho reproducibility
)
print(f"📊 Train: {len(train_data)} | Val: {len(eval_data)}")

# ==========================================
# 4. TRAINING ARGUMENTS (Cải tiến v2)
# ==========================================
OUTPUT_DIR = "/kaggle/working/output/VieNeu-LoRA-v2"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=600,                       # [v2] Giảm từ 1000 (overfit từ step ~400 ở v1)
    per_device_train_batch_size=2,       # [v2] Tăng lên 2 nhờ dynamic padding
    per_device_eval_batch_size=2,  
    gradient_accumulation_steps=4,       # effective batch = 2×4 = 8 (giữ nguyên)
    gradient_checkpointing=True,         
    
    bf16=True,                           # BẬT BF16 (Đồng bộ mô hình gốc)
    fp16=False,                          # TẮT FP16 (Tránh tràn số)
    
    optim="adamw_8bit",                  
    learning_rate=5e-5,                  # [v2] Giảm 50%: 1e-4 → 5e-5
    lr_scheduler_type="cosine",          # [v2] Cosine decay thay linear
    weight_decay=0.05,                   # [v2] Tăng regularization: 0.01 → 0.05
    warmup_steps=60,                     # [v2] 10% of max_steps (thay warmup_ratio)
    
    logging_steps=10,                    # [v2] Log dày hơn để theo dõi
    report_to="none",
    
    save_steps=50,                       # [v2] Save checkpoint thường xuyên hơn
    eval_strategy="steps",
    eval_steps=50,                       # [v2] Eval mỗi 50 steps
    save_total_limit=5,                  # [v2] Giữ 5 checkpoints
    load_best_model_at_end=True,         
    metric_for_best_model="eval_loss",   
    greater_is_better=False,
)

# [v2] Dynamic Padding Collator
dynamic_collator = DynamicPaddingCollator(pad_token_id=tokenizer.pad_token_id)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=train_data, 
    eval_dataset=eval_data, 
    data_collator=dynamic_collator,       # [v2] Dynamic padding thay default_data_collator
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5)  # [v2] Tự dừng khi overfit
    ]
)

# ==========================================
# 5. TỰ ĐỘNG TÌM KÝ ỨC & TRAIN
# ==========================================
last_checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print(f"\n🔄 TÌM THẤY CHECKPOINT! Đang tiếp tục Train từ mốc: {last_checkpoint}...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("\n🚀 Không có Checkpoint. Bắt đầu Train từ con số 0...")
    trainer.train()

# Lưu Model cuối (best model đã được load tự động)
FINAL_DIR = f"{OUTPUT_DIR}-Final"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"🎉 Model LoRA v2 đã được lưu tại: {FINAL_DIR}")

## 🔊 Test Inference — Nghe thử kết quả ngay trên Kaggle

Cell này sinh 3 câu thử nghiệm và phát âm thanh trực tiếp để đánh giá chất lượng.

In [ ]:
import IPython.display as ipd
import numpy as np

# Tải lại model từ checkpoint tốt nhất
from peft import PeftModel

FINAL_DIR = "/kaggle/working/output/VieNeu-LoRA-v2-Final"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
test_model = PeftModel.from_pretrained(base_model, FINAL_DIR)
test_model.eval()

# NeuCodec decoder
codec_test = NeuCodec.from_pretrained("neuphonic/neucodec").to(device)
codec_test.eval()

test_sentences = [
    "Xin chào các bạn Sky yêu thương, hôm nay trời đẹp quá.",
    "Anh rất nhớ các bạn, cảm ơn mọi người đã ở bên anh suốt những năm qua.",
    "Học sinh Việt Nam đạt huy chương vàng quốc tế tại cuộc thi khoa học."
]

print("\n" + "="*60)
print("🔊 BẮT ĐẦU TEST INFERENCE")
print("="*60)

for i, sentence in enumerate(test_sentences):
    print(f"\n📝 Câu {i+1}: {sentence}")
    
    phones = phonemize_with_dict(sentence)
    prompt = f"<|TEXT_PROMPT_START|>{phones}<|TEXT_PROMPT_END|><|SPEECH_GENERATION_START|>"
    
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output = test_model.generate(
            input_ids,
            max_new_tokens=800,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2
        )
    
    # Trích xuất speech codes từ output
    generated_ids = output[0][input_ids.shape[1]:].tolist()
    
    speech_end_id = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_END|>')
    if speech_end_id in generated_ids:
        generated_ids = generated_ids[:generated_ids.index(speech_end_id)]
    
    # Decode token IDs thành speech codes
    speech_codes = []
    for tid in generated_ids:
        token_str = tokenizer.decode([tid]).strip()
        if token_str.startswith('<|speech_') and token_str.endswith('|>'):
            try:
                code = int(token_str.replace('<|speech_', '').replace('|>', ''))
                speech_codes.append(code)
            except ValueError:
                continue
    
    if len(speech_codes) > 10:
        codes_tensor = torch.tensor([speech_codes], dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            wav = codec_test.decode_code(codes_tensor)
        wav_np = wav.squeeze().cpu().numpy()
        
        print(f"   ✅ Sinh được {len(speech_codes)} speech codes → {len(wav_np)/24000:.2f}s audio")
        ipd.display(ipd.Audio(wav_np, rate=24000))
    else:
        print(f"   ⚠️ Chỉ sinh được {len(speech_codes)} codes (quá ít để giải mã)")

print("\n" + "="*60)
print("✅ TEST INFERENCE HOÀN TẤT")
print("="*60)

## 📤 Upload LoRA lên Hugging Face Hub

Chạy cell này để đẩy LoRA adapter lên cloud, giúp tải về máy local dễ dàng.

In [ ]:
from huggingface_hub import HfApi

FINAL_DIR = "/kaggle/working/output/VieNeu-LoRA-v2-Final"
REPO_ID = "phmngcduy/VieNeu-LoRA-SonTungMTP-v2"  # ⚠️ Thay bằng repo của bạn

api = HfApi()

# Tạo repo nếu chưa có
api.create_repo(REPO_ID, exist_ok=True, private=True)

# Upload toàn bộ thư mục LoRA
api.upload_folder(
    folder_path=FINAL_DIR,
    repo_id=REPO_ID,
    commit_message="Upload VieNeu LoRA v2 - Optimized training (dynamic padding, cosine LR, early stopping)"
)

print(f"\n🚀 LoRA v2 đã được upload tại: https://huggingface.co/{REPO_ID}")
print(f"📥 Tải về local bằng lệnh:")
print(f"   git clone https://huggingface.co/{REPO_ID} model/")